In [ ]:
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from pathlib import Path
import pandas as pd
from src.utils import split_dataframe
import os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report
import joblib

tqdm.pandas()
load_dotenv()
SEED = int(os.getenv("SEED", "42"))

## TF-IDF alapú LinearSVC tanítása gyakori ICD-10-CM chapter osztályozásra (Training a TF-IDF-based LinearSVC for frequent ICD-10-CM chapter classification)

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "frequent_chapter/without_dropped_sections",
    processed_data_dir / "frequent_chapter/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")
    
    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))        
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_base_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _  = split_dataframe(df, "chapter", "subject_id", 10, SEED)

In [ ]:
df_train = pd.concat([df_train, df_val])
print(f"Number of rows in the modified train dataset : {len(df_train)}")

X_train_text = df_train["text"]
y_train = mlb.transform(df_train["chapter"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["chapter"])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

In [ ]:
parameters = {
    'tfidf__max_features': [5000, 7500, 10000],
    'tfidf__ngram_range': [(1, 1),(1, 2)],
    'tfidf__max_df': [0.8, 0.9],
    'tfidf__min_df': [0.001, 0.01],
    'clf__estimator__C': [0.1, 1],
    'clf__estimator__class_weight': [None,"balanced"],
    'tfidf__sublinear_tf': [True,False]
}

In [ ]:
mskf = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=mskf,  
    scoring='f1_micro', 
    n_jobs=6,        
)

In [ ]:
grid_search.fit(X_train_text, y_train)

In [ ]:
print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest (cross-validated) micro f1 score on train data:")
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_text)
y_scores = best_model.decision_function(X_test_text)

best_params = {
    "max_features": grid_search.best_params_.get('tfidf__max_features'),
    "ngram_range": grid_search.best_params_.get('tfidf__ngram_range'),
    "min_df": grid_search.best_params_.get('tfidf__min_df'), 
    "max_df": grid_search.best_params_.get('tfidf__max_df'),
    "C": grid_search.best_params_.get('clf__estimator__C'),           
    "class_weight": grid_search.best_params_.get('clf__estimator__class_weight'),
    "sublinear_tf": grid_search.best_params_.get('tfidf__sublinear_tf'),
}

metrics = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_micro_f1": f1_score(y_test, y_pred, average='micro'),
    "test_macro_f1": f1_score(y_test, y_pred, average='macro'),
    "test_samples_f1": f1_score(y_test, y_pred, average='samples'),
    "test_weighted_f1": f1_score(y_test, y_pred, average='weighted'),
    "test_micro_precision": precision_score(y_test, y_pred, average='micro', zero_division=0),
    "test_macro_precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "test_micro_recall": recall_score(y_test, y_pred, average='micro'),
    "test_macro_recall": recall_score(y_test, y_pred, average='macro'),
    "test_micro_auc": roc_auc_score(y_test, y_scores, average='micro'),
    "test_macro_auc": roc_auc_score(y_test, y_scores, average='macro'),
}

report = classification_report(
    y_test, 
    y_pred, 
    target_names=mlb.classes_, 
    zero_division=0,
    output_dict= True
)

report = pd.DataFrame(report).transpose()
summary_metrics = ["micro avg", "macro avg", "weighted avg", "samples avg"]

main_report = report.drop(index=[i for i in summary_metrics if i in report.index])
summary_report = report.loc[[i for i in summary_metrics if i in report.index]]

main_report = main_report.sort_values(by="f1-score", ascending=False)

report = pd.concat([main_report, summary_report])

print(metrics)
print(report)

In [ ]:
eval_dir = baseline_model_path / "evaluation_results"
eval_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([best_params]).to_json(eval_dir / "best_parameters.json", orient='records', indent=1)

pd.DataFrame([metrics]).to_json(eval_dir / "metrics.json", orient='records', indent=1)

report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
best_model_dir = baseline_model_path / "best_model"
best_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, best_model_dir / "pipeline.joblib")

joblib.dump(mlb, best_model_dir / "mlb.joblib")

## TF-IDF alapú LinearSVC tanítása top 50 ICD-10-CM kód osztályozásra (Training a TF-IDF-based LinearSVC for top 50 ICD-10-CM code classification)

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "top_50_code/without_dropped_sections",
    processed_data_dir / "top_50_code/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")

    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_base_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _ = split_dataframe(df, "icd_code", "subject_id", 10, SEED)

In [ ]:
df_train = pd.concat([df_train, df_val])
print(f"Number of rows in the modified train dataset : {len(df_train)}")

X_train_text = df_train["text"]
y_train = mlb.transform(df_train["icd_code"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["icd_code"])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

In [ ]:
parameters = {
    'tfidf__max_features': [5000, 7500, 10000],
    'tfidf__ngram_range': [(1, 1),(1, 2)],
    'tfidf__max_df': [0.8, 0.9],
    'tfidf__min_df': [0.001, 0.01],
    'clf__estimator__C': [0.1, 1],
    'clf__estimator__class_weight': [None,"balanced"],
    'tfidf__sublinear_tf': [True,False]
}

In [ ]:
mskf = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=mskf,
    scoring='f1_micro',
    n_jobs=6,
)

In [ ]:
grid_search.fit(X_train_text, y_train)

In [ ]:
print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest (cross-validated) micro f1 score on train data:")
print(grid_search.best_score_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_text)
y_scores = best_model.decision_function(X_test_text)

best_params = {
    "max_features": grid_search.best_params_.get('tfidf__max_features'),
    "ngram_range": grid_search.best_params_.get('tfidf__ngram_range'),
    "min_df": grid_search.best_params_.get('tfidf__min_df'),
    "max_df": grid_search.best_params_.get('tfidf__max_df'),
    "C": grid_search.best_params_.get('clf__estimator__C'),
    "class_weight": grid_search.best_params_.get('clf__estimator__class_weight'),
    "sublinear_tf": grid_search.best_params_.get('tfidf__sublinear_tf'),
}

metrics = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_micro_f1": f1_score(y_test, y_pred, average='micro'),
    "test_macro_f1": f1_score(y_test, y_pred, average='macro'),
    "test_samples_f1": f1_score(y_test, y_pred, average='samples'),
    "test_weighted_f1": f1_score(y_test, y_pred, average='weighted'),
    "test_micro_precision": precision_score(y_test, y_pred, average='micro', zero_division=0),
    "test_macro_precision": precision_score(y_test, y_pred, average='macro', zero_division=0),
    "test_micro_recall": recall_score(y_test, y_pred, average='micro'),
    "test_macro_recall": recall_score(y_test, y_pred, average='macro'),
    "test_micro_auc": roc_auc_score(y_test, y_scores, average='micro'),
    "test_macro_auc": roc_auc_score(y_test, y_scores, average='macro'),
}

report = classification_report(
    y_test,
    y_pred,
    target_names=mlb.classes_,
    zero_division=0,
    output_dict=True
)

report = pd.DataFrame(report).transpose()
summary_metrics = ["micro avg", "macro avg", "weighted avg", "samples avg"]

main_report = report.drop(index=[i for i in summary_metrics if i in report.index])
summary_report = report.loc[[i for i in summary_metrics if i in report.index]]

main_report = main_report.sort_values(by="f1-score", ascending=False)

report = pd.concat([main_report, summary_report])

print(metrics)
print(report)

In [ ]:
eval_dir = baseline_model_path / "evaluation_results"
eval_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([best_params]).to_json(eval_dir / "best_parameters.json", orient='records', indent=1)

pd.DataFrame([metrics]).to_json(eval_dir / "metrics.json", orient='records', indent=1)

report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
best_model_dir = baseline_model_path / "best_model"
best_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, best_model_dir / "pipeline.joblib")

joblib.dump(mlb, best_model_dir / "mlb.joblib")